# 05 Statistics That Hold Up
### Confidence intervals, subgroups, และ calibration ก่อนส่ง deploy

Notebook นี้พาคุณตรวจโมเดลก่อนปล่อยจริง ไม่ใช่แค่ดู accuracy ตัวเดียว แต่ดูว่าตัวเลขนั้น "ยืนได้จริง" หรือไม่: มี confidence interval ไหม ทำงานดีเท่ากันในทุกกลุ่มประชากรไหม และ probability ที่ทำนายออกมาเชื่อถือได้แค่ไหน

> 🎯 **เป้าหมาย:** คำนวณ AUROC พร้อม bootstrap CI, ตรวจ subgroup fairness, และอ่าน calibration curve

## 1. ข้อมูลจำลอง: ความเสี่ยงกลับมารักษาซ้ำใน 30 วัน

เราสร้างชุดข้อมูลสังเคราะห์แทนข้อมูลผู้ป่วยจริง เพื่อฝึกโดยไม่แตะข้อมูลจริง ตามหลักที่เรียนใน Basics

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
n = 1200

age = rng.normal(58, 15, n).clip(18, 95)
num_admissions = rng.poisson(1.4, n)
group = rng.choice(["group_a", "group_b"], size=n, p=[0.7, 0.3])

# ความเสี่ยงจริง (แฝงอยู่) ขึ้นกับอายุและจำนวนครั้งที่เคยแอดมิต บวกสัญญาณรบกวนเล็กน้อยตามกลุ่ม
logit = -3 + 0.03 * age + 0.6 * num_admissions + np.where(group == "group_b", -0.4, 0)
prob_true = 1 / (1 + np.exp(-logit))
readmitted = rng.binomial(1, prob_true)

df = pd.DataFrame({"age": age, "num_admissions": num_admissions, "group": group, "readmitted": readmitted})
df["group"] = df["group"].map({"group_a": "Group A", "group_b": "Group B"})
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,1200.0,NaN,NaN,NaN,57.658243,14.647865,18.0,47.767595,58.051786,67.231561,95.0
num_admissions,1200.0,NaN,NaN,NaN,1.370833,1.227289,0.0,0.0,1.0,2.0,7.0
group,1200,2,Group A,875,NaN,NaN,NaN,NaN,NaN,NaN,NaN
readmitted,1200.0,NaN,NaN,NaN,0.383333,0.486401,0.0,0.0,0.0,1.0,1.0


## 2. Train แบบง่าย แล้วทำนายบน test set

In [2]:
X = df[["age", "num_admissions"]]
y = df["readmitted"]
X_train, X_test, y_train, y_test, group_train, group_test = train_test_split(
    X, y, df["group"], test_size=0.3, stratify=y, random_state=7)

model = LogisticRegression().fit(X_train, y_train)
proba_test = model.predict_proba(X_test)[:, 1]
print(f"Test set: {len(X_test)} ราย, อัตราการกลับมารักษาซ้ำจริง {y_test.mean():.1%}")

Test set: 360 ราย, อัตราการกลับมารักษาซ้ำจริง 38.3%


## 3. AUROC พร้อม bootstrap confidence interval

AUROC ตัวเดียวบอกไม่พอ เราต้อง resample ข้อมูล test ซ้ำหลายพันครั้งเพื่อดูว่าตัวเลขนี้ "แกว่ง" แค่ไหน นี่คือสิ่งที่ dashboard ใน Deployment ควรแสดงคู่กับทุกตัวเลขหลัก

In [3]:
from sklearn.metrics import roc_auc_score

def bootstrap_auc(y_true, y_score, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    scores = []
    idx_all = np.arange(len(y_true))
    for _ in range(n_boot):
        idx = rng.choice(idx_all, size=len(idx_all), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        scores.append(roc_auc_score(y_true[idx], y_score[idx]))
    return np.array(scores)

boot_scores = bootstrap_auc(y_test, proba_test)
point = roc_auc_score(y_test, proba_test)
lo, hi = np.percentile(boot_scores, [2.5, 97.5])
print(f"AUROC = {point:.3f}  (95% CI {lo:.3f} to {hi:.3f}, n={len(y_test)})")

AUROC = 0.716  (95% CI 0.660 to 0.770, n=360)


```{important}
เขียนแค่ "AUROC 0.79" อ่านเหมือนความแน่นอนที่ไม่มีจริง เขียน "AUROC 0.79 (95% CI 0.74 to 0.84, n=360)" บอกความจริงมากกว่า และเป็นสิ่งที่ dashboard ใน [Deployment](../curriculum/deployment.html) ต้องแสดงเสมอ
```

## 4. ตรวจ subgroup: โมเดลทำงานดีเท่ากันในทุกกลุ่มไหม

โมเดลที่แม่นเฉลี่ยดี อาจแม่นไม่เท่ากันในแต่ละกลุ่มประชากร ต้องแยกดูเสมอ ไม่ใช่ดูแค่ค่าเฉลี่ยรวม

In [4]:
rows = []
for g in sorted(group_test.unique()):
    mask = (group_test == g).to_numpy()
    auc_g = roc_auc_score(y_test[mask], proba_test[mask])
    rows.append({"group": g, "n": int(mask.sum()), "positive_rate": float(y_test[mask].mean()), "auroc": round(auc_g, 3)})

subgroup_report = pd.DataFrame(rows)
subgroup_report

,group,n,positive_rate,auroc
0,Group A,260,0.40,0.737
1,Group B,100,0.34,0.670


## 5. Calibration: เมื่อโมเดลบอกความเสี่ยง 20% มันแปลว่าอะไรจริง ๆ

AUROC วัดการจัดอันดับ แต่ไม่ได้บอกว่าตัวเลข probability ที่ออกมาเชื่อถือได้ไหม เราแบ่งผู้ป่วยเป็นกลุ่มตามความเสี่ยงที่ทำนาย แล้วเทียบกับอัตราที่เกิดขึ้นจริงในแต่ละกลุ่ม

In [5]:
from sklearn.calibration import calibration_curve

frac_pos, mean_pred = calibration_curve(y_test, proba_test, n_bins=5, strategy="quantile")
calib = pd.DataFrame({
    "predicted_risk": np.round(mean_pred, 3),
    "observed_rate": np.round(frac_pos, 3),
})
calib["gap"] = np.round(calib["observed_rate"] - calib["predicted_risk"], 3)
calib

,predicted_risk,observed_rate,gap
0,0.176,0.181,0.005
1,0.260,0.250,-0.010
2,0.352,0.333,-0.019
3,0.469,0.500,0.031
4,0.674,0.653,-0.021


ถ้า `gap` ใกล้ 0 ทุกแถว โมเดล calibrate ดี ถ้าห่างมากในบางแถว โมเดลมั่นใจเกินจริงหรือน้อยเกินไปในช่วงความเสี่ยงนั้น ซึ่งอันตรายกว่าที่คิด เพราะแพทย์มักเชื่อตัวเลขที่โมเดลให้ตรง ๆ

## 6. หน้าตาตอน deploy จริง (โค้ดตัวอย่าง ไม่ต้องรัน)

ตัวเลขข้างบนคือสิ่งที่ควรโผล่บน dashboard ทุกครั้ง ไม่ใช่แค่ตัวเลขเดี่ยว ๆ:

```python
import streamlit as st

st.metric("ความเสี่ยงกลับมารักษาซ้ำใน 30 วัน", f"{proba_test[0]:.0%}",
          help=f"AUROC {point:.2f} (95% CI {lo:.2f}-{hi:.2f}) / กลุ่ม A vs B ดูตาราง subgroup ด้านบน")
st.caption("Model v0.1 / สำหรับการพิจารณาของแพทย์เท่านั้น ไม่ใช่การวินิจฉัยอัตโนมัติ")
```

อ่านเรื่อง FastAPI, Docker, และการเลือก cloud หรือ on-premise เต็ม ๆ ใน [Deployment](../curriculum/deployment.html)

## สรุปและก้าวต่อไป

- AUROC เดี่ยว ๆ ไม่พอ ต้องมี bootstrap confidence interval กำกับเสมอ
- ตรวจ subgroup แยก ไม่ใช่ดูแค่ค่าเฉลี่ยรวม โมเดลนี้ทำงานต่างกันเล็กน้อยระหว่างกลุ่ม A และ B
- Calibration บอกว่าตัวเลข probability เชื่อถือได้แค่ไหน ไม่ใช่แค่การจัดอันดับถูก

**ลองต่อ:** เปลี่ยน `n_bins` ใน calibration_curve เป็น 10 แล้วดูว่า gap เปลี่ยนไปอย่างไร จากนั้นอ่าน [Strategy and Governance](../curriculum/governance.html) เพื่อดูว่าตัวเลขเหล่านี้ต้องรายงานอย่างไรในเอกสารกำกับดูแล